In [ ]:
import numpy as np
import matplotlib.pylab as plt

In [ ]:
# A Hamiltonian version of our evolve() function, where we keep (generalized) positions
# and momenta in separate "q" and "p" variables, and the derivatives are computed with
# "qdot" and "pdot" functions that compute partial derivatives of the Hamiltonian; and
# they share the "H_kwargs" keyword arguments.  "step" is a stepping function, as before,
# but its function signature is a little different.
def evolve_hamiltonian(t0, t1, q0, p0, N, qdot, pdot, H_kwargs, step):
    # create the time grid
    t = np.linspace(t0, t1, N+1)
    # pre-allocate space for the q,p values at each step
    Nq = len(q0)
    Np = len(p0)
    q = np.zeros((N+1, Nq))
    p = np.zeros((N+1, Np))
    # initialize the state
    q[0,:] = q0
    p[0,:] = p0
    for i in range(N):
        # time step size (constant in this function)
        h = t[i+1] - t[i]
        # use the step function to step forward!
        q[i+1, :], p[i+1, :] = step(t[i], q[i,:], p[i,:], h, qdot, pdot, H_kwargs)
    return t, q, p

In [ ]:
### FIXME -- finish writing this function
def qdot_kepler(t, q, p, GM=1., m=1.):
    # Here we compute and return the partial derivative  dH/dp  for the Kepler orbit problem.
    return #...

In [ ]:
### FIXME -- finish writing this function
def pdot_kepler(t, q, p, GM=1., m=1.):
    # Here we compute and return the partial derivative  -dH/dq  for the Kepler orbit problem.
    return #...

In [ ]:
# The Forward Euler method, in its Hamiltonian version
def step_forward_euler(t, q, p, h, qdot, pdot, H_kwargs):
    dq = qdot(t, q, p, **H_kwargs)
    dp = pdot(t, q, p, **H_kwargs)
    q_next = q + dq * h
    p_next = p + dp * h
    return q_next, p_next

In [ ]:
# Midpoint method, Hamiltonian version
def step_midpoint(t, q, p, h, qdot, pdot, H_kwargs):
    dq = qdot(t, q, p, **H_kwargs)
    dp = pdot(t, q, p, **H_kwargs)
    q_half = q + dq * h/2.
    p_half = p + dp * h/2.
    t_half = t +      h/2.
    dq_half = qdot(t_half, q_half, p_half, **H_kwargs)
    dp_half = pdot(t_half, q_half, p_half, **H_kwargs)
    q_next = q + dq_half * h
    p_next = p + dp_half * h
    return q_next, p_next

In [ ]:
# Runge-Kutta 4th-order, Hamiltonian version
def step_rk4(t, q, p, h, qdot, pdot, H_kwargs):
    # RK4 evaluates the derivatives at 4 points
    # slope 1: like Forward Euler
    dq1 = qdot(t, q, p, **H_kwargs)
    dp1 = pdot(t, q, p, **H_kwargs)
    # point 1: step half-way with slope 1
    t1 = t + h/2.
    p1 = p + dp1 * h/2.
    q1 = q + dq1 * h/2.

    # slope 2: like RK2 or midpoint: evaluate the slope half-way through the time interval.
    dq2 = qdot(t1, q1, p1, **H_kwargs)
    dp2 = pdot(t1, q1, p1, **H_kwargs)
    # point 2: step half-way, but using slope 2
    t2 = t + h/2.
    p2 = p + dp2 * h/2.
    q2 = q + dq2 * h/2.

    # slope 3: evaluate at point 2.
    dq3 = qdot(t2, q2, p2, **H_kwargs)
    dp3 = pdot(t2, q2, p2, **H_kwargs)
    # point 3: move to the end-point of the interval with slope 3.
    t3 = t + h
    q3 = q + dq3 * h
    p3 = p + dp3 * h

    # slope 4: evaluate at point 3.
    dq4 = qdot(t3, q3, p3, **H_kwargs)
    dp4 = pdot(t3, q3, p3, **H_kwargs)

    # RK4 combines those 4 estimates of the slope:
    dq = 1./6. * (dq1 + 2.*dq2 + 2.*dq3 + dq4)
    dp = 1./6. * (dp1 + 2.*dp2 + 2.*dp3 + dp4)
    q_next = q + dq * h
    p_next = p + dp * h
    return q_next, p_next

In [ ]:
# Semi-implicit Euler, Hamiltonian version
def step_semi_implicit_euler(t, q, p, h, qdot, pdot, H_kwargs):
    # First, find "p" at the endpoint using the Forward Euler prediction; call that "p1"
    dp = pdot(t, q, p, **H_kwargs)
    p1 = p + dp * h
    # Use that "p1" to find "dq"
    dq1 = qdot(t, q, p1, **H_kwargs)
    q1 = q + dq1 * h
    return q1, p1

In [ ]:
# add your step_yoshida function...

In [ ]:
# Initialize our Kepler problem
GM = 1.
m = 1.
kepler_kwargs = dict(GM=GM, m=m)
q0 = np.array([1.0, 0.0, 0.0])
p0 = np.array([0.0, 0.9, 0.0])

In [ ]:
# Compute some properties of the orbit that will be useful later -
# kinetic energy
K0 = np.sum(p0**2) / (2.*m)
# potential energy
U0 = -GM * m / np.sqrt(np.sum(q0**2))
# sum of energy
E0 = K0 + U0
# semi-major axis
a = -GM * m / (2. * E0)
# period
period = 2.*np.pi * np.sqrt(a**3 * GM)

In [ ]:
# Let's run 4 orbits, with each of the stepping functions...
t_start, t_end = 0., period * 4
N = 1000

methods = [
    ('Forward Euler',   step_forward_euler),
    ('Midpoint',        step_midpoint),
    ('RK4',             step_rk4),
    ('Semi-impl Euler', step_semi_implicit_euler),
    # When your function is done, you can add it here!
    #('Yoshida',         step_yoshida),
]

# We'll save our t,q,p results for each method in this dictionary:
tqp = {}
# Run each method, using its "step" function.
for name, step in methods:
    t, q, p = evolve_hamiltonian(t_start, t_end, q0, p0, N, qdot_kepler, pdot_kepler, kepler_kwargs,
                                 step)
    # save results in "tqp" dictionary
    tqp[name] = (t, q, p)

In [ ]:
# Plot the orbits
# sun
plt.plot(0, 0, 'yo', ms=20)
for name, (t,q,p) in tqp.items():
    # orbit
    plt.plot(q[:,0], q[:,1], '-', label=name)
    # start point
    plt.plot(q[0,0], q[0,1], 'o')
    # end point
    plt.plot(q[-1,0], q[-1,1], 'x')
plt.axis('square');
plt.legend();

In [ ]:
# Plot the energies, relative to the initial energy E0, for each method.
plt.figure(figsize=(10,5))
plt.subplots_adjust(hspace=0.3, wspace=0.3)
for i, (name, (t,q,p)) in enumerate(tqp.items()):
    K = np.sum(p**2, axis=1) / (2.*m)
    U = -GM / np.sqrt(np.sum(q**2, axis=1))
    plt.subplot(2,3, 1+i)
    plt.plot(t, K+U - E0, '-')
    plt.title(name)
# go back and label the first subplot
plt.subplot(2,3,1)
plt.ylabel('Change in total energy')

In [ ]:
# Error/convergence behavior.
# Let's look at the position error at the endpoint after running 4 cycles.  We know the answer should be q0.
q_true = q0

# The number of steps to take.
NN = np.logspace(base=10, start=2, stop=5, num=10).astype(int)
# The errors for each method
errors = {}
for name,_ in methods:
    errors[name] = np.zeros(len(NN))

# This may take ~ 30 seconds to run
for i,N in enumerate(NN):
    for name,step_func in methods:
        t, q, p = evolve_hamiltonian(t_start, t_end, q0, p0, N, qdot_kepler, pdot_kepler, kepler_kwargs,
                                     step_func)
        err = np.sqrt(np.sum((q[-1,:] - q_true)**2))
        errors[name][i] = err

In [ ]:
# Convergence plots
for name,err in errors.items():
    err = errors[name]
    plt.loglog(NN, err, '.-', label=name)
plt.ylabel('Error after 4 orbits')
plt.xlabel('Number of steps (~1/h)')
plt.legend();

In [ ]:
# Let's look at the energy after running for a long time
t_long = t_end * 100
N = 10000
# The energy for each method
energies = {}
# The solutions for each method
tqp = {}

#R0 = np.sqrt(np.sum(q0**2))
#E0 = -GM * m / R0 + np.sum(p0**2) / (2.*m)

for name,step_func in methods:
    t, q, p = evolve_hamiltonian(t_start, t_long, q0, p0, N, qdot_kepler, pdot_kepler, kepler_kwargs,
                                 step_func)
    R = np.sqrt(np.sum(q**2, axis=1))
    energies[name] = -GM * m / R + np.sum(p**2, axis=1) / (2.*m)
    tqp[name] = (t,q,p)

In [ ]:
# Plot the absolute error in the energy compared to the initial energy, in log space
for name,energy in energies.items():
    plt.semilogy(t, np.abs(energy - E0), '-', label=name, alpha=0.5)
plt.ylabel('Error in energy')
plt.xlabel('Step number (~time)')
plt.legend();